# SPHINX Injection Analysis

This notebook mirrors `Retrieval_analysis_contam.ipynb`, but it reads the SPHINX-injected observations from `../observations_sphinx/`. The retrieval products are still interpreted with PHOENIX stellar models.

# POSEIDON Retrieval — Stellar-contaminated case (two_spots)

This notebook runs a POSEIDON retrieval for a **stellar contamination** scenario using
`stellar_contam="two_spots"`.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

from POSEIDON.constants import M_E, R_E, R_Sun
from POSEIDON.core import (
    create_planet,
    create_star,
    define_model,
    load_data,
    read_opacities,
    set_priors,
    wl_grid_constant_R,
)
from POSEIDON.corner import generate_cornerplot
from POSEIDON.instrument import bin_spectrum_to_data
from POSEIDON.stellar import (
    precompute_stellar_spectra,
    stellar_contamination,
)
from POSEIDON.utility import read_data, read_retrieved_spectrum, plot_collection
from POSEIDON.visuals import plot_data, plot_spectra_retrieved


# -----------------------------
# Case configuration (edit as needed)
# -----------------------------
PLANET_NAME = "Trappist-1e"
DATA_DIR = Path("../observations_sphinx")
INSTRUMENTS = ["JWST_NIRSpec_PRISM"]

OBSERVATION_FILE = "pandexo_output_10transits_sphinx_fspot0.26_ffac0.70.dat"
MODEL_NAME = "sphinx_contam_10T_0.26spot-0.70fac"

BULK_SPECIES = ["N2"]
PARAM_SPECIES = ["H2O", "CH4", "CO2", "O3"]

CLEAN_PATH = Path("../pandexo_spec.txt")
LOG_PATH = Path("chi2_log_sphinx_contam.csv")

# Model wavelength grid
WL_MIN_UM = 0.4
WL_MAX_UM = 6.0
R_GRID = 4000

# Retrieval bookkeeping
N_FREE_PARAMS = 11


## 1) Star and planet definitions


In [ ]:
# -----------------------------
# Star definition (TRAPPIST-1 baseline)
# -----------------------------
R_s = 0.1192 * R_Sun
T_s = 2566.0
met_s = 0.00
log_g_s = 5.2396

star = create_star(
    R_s,
    T_s,
    log_g_s,
    met_s,
    stellar_grid="phoenix",
)


# -----------------------------
# Planet definition (TRAPPIST-1e)
# -----------------------------
R_p = 0.917985 * R_E
M_p = 0.6356 * M_E
T_eq = 255.0

planet = create_planet(PLANET_NAME, R_p, mass=M_p, T_eq=T_eq)


## 2) Load dataset and visualize


In [ ]:
wl_model = wl_grid_constant_R(WL_MIN_UM, WL_MAX_UM, R_GRID)

data = load_data(
    str(DATA_DIR),
    datasets=[OBSERVATION_FILE],
    instruments=INSTRUMENTS,
    wl_model=wl_model,
)

fig_data = plot_data(data, PLANET_NAME)


## 3) Define the contaminated stellar model (two_spots) and priors


In [ ]:
model = define_model(
    MODEL_NAME,
    BULK_SPECIES,
    PARAM_SPECIES,
    PT_profile="isotherm",
    cloud_model="cloud-free",
    stellar_contam="two_spots",
)

print("Free parameters:", model["param_names"])


prior_types = {
    "T": "uniform",
    "R_p_ref": "uniform",
    "log_H2O": "uniform",
    "log_CH4": "uniform",
    "log_CO2": "uniform",
    "log_O3": "uniform",
    "f_spot": "uniform",
    "f_fac": "uniform",
    "T_phot": "uniform",
    "T_fac": "uniform",
    "T_spot": "uniform",
}

prior_ranges = {
    "T": [200, 400],
    "R_p_ref": [0.85 * R_p, 1.15 * R_p],
    "log_H2O": [-8, -1],
    "log_CH4": [-8, -1],
    "log_CO2": [-5, -1],
    "log_O3": [-8, -1],
    "f_spot": [0.0, 0.26],
    "f_fac": [0.0, 0.70],
    "T_phot": [0.9 * T_s, 1.1 * T_s],
    "T_fac": [T_s, T_s + 15.0],
    "T_spot": [0.8 * T_s, 0.9 * T_s],
}

priors = set_priors(planet, star, model, data, prior_types, prior_ranges)


## 4) Read opacities and precompute stellar spectra

The `precompute_stellar_spectra` call is required for `two_spots` contamination.


In [ ]:
opacity_treatment = "opacity_sampling"

T_fine = np.arange(200, 401, 10)
log_P_fine = np.arange(-2, 2.0001, 0.2)

opac = read_opacities(model, wl_model, opacity_treatment, T_fine, log_P_fine)

""" precompute_stellar_spectra(
    comm=1,
    wl_out=wl_model,
    star=star,
    prior_types=prior_types,
    prior_ranges=prior_ranges,
    stellar_contam="two_spots",
    T_step_interp=20,
) """


## 5) Plot retrieved transmission spectrum and corner plot

This assumes retrieval outputs exist for `(PLANET_NAME, MODEL_NAME)`.


In [ ]:
(
    wl_out,
    spec_low2,
    spec_low1,
    spec_median,
    spec_high1,
    spec_high2,
) = read_retrieved_spectrum(PLANET_NAME, MODEL_NAME)

spectra_median = plot_collection(spec_median, wl_out, collection=[])
spectra_low1 = plot_collection(spec_low1, wl_out, collection=[])
spectra_low2 = plot_collection(spec_low2, wl_out, collection=[])
spectra_high1 = plot_collection(spec_high1, wl_out, collection=[])
spectra_high2 = plot_collection(spec_high2, wl_out, collection=[])

fig_spec = plot_spectra_retrieved(
    spectra_median,
    spectra_low2,
    spectra_low1,
    spectra_high1,
    spectra_high2,
    PLANET_NAME,
    data,
    R_to_bin=100,
    data_labels=["NIRSpec_PRISM"],
    data_colour_list=["lime"],
)

fig_corner = generate_cornerplot(planet, model)


## 6) Manual update step: re-create the star with retrieved values (IMPORTANT)

**This cell is intentionally manual.**

After inspecting the plots (or POSEIDON output files), update the following values:
- `T_s_retrieved` (photosphere),
- `f_spot_retrieved`, `T_spot_retrieved`,
- `f_fac_retrieved`, `T_fac_retrieved`.

Then compute the contamination factor $\epsilon(\lambda)$.


In [ ]:
# -----------------------------
# IMPORTANT: manually update these after inspecting retrieval outputs
# -----------------------------
T_s_retrieved = 2641.0 # Τ_phot
f_spot_retrieved = 0.00
T_spot_retrieved = 2185.0
f_fac_retrieved = 0.11
T_fac_retrieved = 2653.0

star_retrieved = create_star(
    R_s,
    T_s_retrieved,
    log_g_s,
    met_s,
    stellar_grid="phoenix",
    stellar_contam="two_spots",
    f_spot=f_spot_retrieved,
    T_spot=T_spot_retrieved,
    f_fac=f_fac_retrieved,
    T_fac=T_fac_retrieved,
)

epsilon = stellar_contamination(star_retrieved, wl_out)
epsilon = np.asarray(epsilon, dtype=float)

print("Computed epsilon(lambda) for contamination correction.")


## 7) Metrics vs clean truth using contamination-corrected spectrum


In [ ]:
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd


def load_clean_two_cols(path: str | Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load a two-column clean spectrum file: (wl_um, depth).

    Lines starting with '#' are ignored. Output is sorted by wavelength.
    """
    arr = np.genfromtxt(path, comments="#", dtype=float)

    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)

    if arr.shape[1] < 2:
        raise ValueError(
            "Expected >= 2 columns (wl_um, depth) in the clean spectrum file."
        )

    wl_clean = arr[:, 0].astype(float)
    y_clean = arr[:, 1].astype(float)

    idx = np.argsort(wl_clean)
    return wl_clean[idx], y_clean[idx]


def bin_average_with_halfbins(
    wl_src: np.ndarray,
    y_src: np.ndarray,
    centers: np.ndarray,
    halfwidths: np.ndarray,
    nsamp: int = 256,
) -> np.ndarray:
    """
    Band-average y_src over [c-h, c+h] via trapezoidal integration
    of a linear interpolation.
    """
    wl_src = np.asarray(wl_src, dtype=float)
    y_src = np.asarray(y_src, dtype=float)
    centers = np.asarray(centers, dtype=float)
    halfwidths = np.asarray(halfwidths, dtype=float)

    sort_idx = np.argsort(wl_src)
    wl_sorted = wl_src[sort_idx]
    y_sorted = y_src[sort_idx]

    out = np.empty_like(centers, dtype=float)

    for i, (c, h) in enumerate(zip(centers, halfwidths)):
        a = c - h
        b = c + h
        x = np.linspace(a, b, nsamp)
        yx = np.interp(x, wl_sorted, y_sorted)
        out[i] = np.trapz(yx, x) / (b - a)

    return out


def compute_metrics_vs_clean(
    clean_path: str | Path,
    data_dir: str | Path,
    observation_file: str,
    instruments,
    wl_model: np.ndarray,
    spec_median: np.ndarray,
    epsilon: np.ndarray,
    n_free_params: int,
) -> dict:
    """
    Compute goodness-of-fit metrics between the contamination-corrected
    retrieved spectrum and the clean truth spectrum binned to the data grid.
    """
    wl_clean, y_clean = load_clean_two_cols(clean_path)

    wl_data, half_bin, y_obs, err_obs = read_data(str(data_dir), observation_file)

    data = load_data(
        str(data_dir),
        datasets=[observation_file],
        instruments=instruments,
        wl_model=wl_model,
    )

    print(f"N observed points: {len(wl_data)}")
    print(f"wl_data range    : {wl_data.min():.4f}–{wl_data.max():.4f} µm")

    median_corrected = np.asarray(spec_median, dtype=float) / np.asarray(epsilon, dtype=float)

    model_binned = bin_spectrum_to_data(median_corrected, wl_model, data)
    clean_binned = bin_average_with_halfbins(wl_clean, y_clean, wl_data, half_bin)

    if not (len(model_binned) == len(clean_binned) == len(err_obs)):
        raise ValueError("Binned arrays and uncertainties must have the same length.")

    sig = np.asarray(err_obs, dtype=float)
    if np.any(sig <= 0) or np.any(~np.isfinite(sig)):
        raise ValueError(
            "err_obs contains invalid uncertainties (non-positive or non-finite)."
        )

    yhat = np.asarray(model_binned, dtype=float)
    y = np.asarray(clean_binned, dtype=float)
    resid = yhat - y

    n_points = int(resid.size)
    p = int(n_free_params)
    dof = int(max(n_points - p, 0))

    chi2 = float(np.sum((resid / sig) ** 2))
    chi2_reduced = float(chi2 / dof) if dof > 0 else np.nan

    mse = float(np.mean(resid**2))
    rmse = float(np.sqrt(mse))
    rmse_ppm = float(1e6 * rmse)

    print("---- METRICS vs CLEAN TRUTH (corrected median / epsilon) ----")
    print(f"N points     : {n_points}")
    print(f"Params p     : {p}")
    print(f"DoF          : {dof}")
    print(f"MSE          : {mse:.6e}")
    print(f"RMSE         : {rmse:.6e}  ({rmse_ppm:.2f} ppm)")
    print(f"chi2         : {chi2:.6f}")
    print(f"chi2_reduced : {chi2_reduced:.6f}")

    return {
        "N": n_points,
        "p": p,
        "dof": dof,
        "MSE": mse,
        "chi2": chi2,
        "chi2_reduced": chi2_reduced,
        "rmse": rmse,
        "rmse_ppm": rmse_ppm,
    }


def append_metrics_to_csv(
    log_path: str | Path,
    planet_name: str,
    model_name: str,
    observation: str | None,
    metrics: dict,
) -> pd.DataFrame:
    """
    Append one metrics row to the CSV log using a fixed column order.
    """
    log_path = Path(log_path)

    columns = [
        "planet_name",
        "model_name",
        "dof",
        "chi2",
        "chi2_reduced",
        "N",
        "p",
        "MSE",
        "observation",
    ]

    row = {
        "planet_name": str(planet_name),
        "model_name": str(model_name),
        "dof": int(metrics["dof"]),
        "chi2": float(metrics["chi2"]),
        "chi2_reduced": float(metrics["chi2_reduced"]),
        "N": int(metrics["N"]),
        "p": int(metrics["p"]),
        "MSE": float(metrics["MSE"]),
        "observation": "" if observation is None else str(observation),
    }

    new_row_df = pd.DataFrame([row], columns=columns)

    if log_path.exists():
        df_log = pd.read_csv(log_path)

        for col in columns:
            if col not in df_log.columns:
                df_log[col] = ""

        df_log = df_log[columns]
        df_log = pd.concat([df_log, new_row_df], ignore_index=True)
    else:
        df_log = new_row_df

    df_log.to_csv(log_path, index=False, float_format="%.10g")
    print(f"Appended row to: {log_path.resolve()}")

    return df_log


# ============================================================
# MAIN
# ============================================================

metrics = compute_metrics_vs_clean(
    clean_path=CLEAN_PATH,
    data_dir=DATA_DIR,
    observation_file=OBSERVATION_FILE,
    instruments=INSTRUMENTS,
    wl_model=wl_model,
    spec_median=spec_median,
    epsilon=epsilon,
    n_free_params=N_FREE_PARAMS,
)

df_log = append_metrics_to_csv(
    log_path="chi2_log_sphinx_contam.csv",
    planet_name=PLANET_NAME,
    model_name=MODEL_NAME,
    observation=OBSERVATION_FILE,
    metrics=metrics,
)